In [ ]:
from pathlib import Path
import os

# Set PROJECT_DATA_DIR before launching Jupyter to use data stored elsewhere.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".gitignore").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = Path(os.environ.get("PROJECT_DATA_DIR", str(PROJECT_ROOT / "data"))).expanduser().resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)


## IMPORTS AND SETTINGS

In [ ]:
#pip install pandas numpy scikit-learn lightgbm tensorflow matplotlib seaborn pyarrow joblib

In [ ]:
# pip install jupyter ipykernel

In [ ]:
import os
import gc
import json
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_fscore_support,
    confusion_matrix
)
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

import lightgbm as lgb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

import joblib

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

pd.set_option("display.max_columns", 500)

## LOAD DATASET

In [ ]:
df = pd.read_csv(str(DATA_DIR / 'train_merged_submission.csv'))
print(df.head(5))
print(df.shape)

## LGBM

In [ ]:
TARGET = "isFraud"
ID_COLS = ["TransactionID"]

In [ ]:
df = df.sort_values("TransactionDT").reset_index(drop=True)

split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx].copy()
valid_df = df.iloc[split_idx:].copy()

print(train_df.shape, valid_df.shape)
print(train_df["isFraud"].mean(), valid_df["isFraud"].mean())

In [ ]:
categorical = []
numerical = []

known_cat_exact = {
    "ProductCD",
    "card1", "card2", "card3", "card4", "card5", "card6",
    "addr1", "addr2",
    "P_emaildomain", "R_emaildomain",
    "DeviceType", "DeviceInfo"
}

for col in train_df.columns:
    if col == "isFraud":
        continue

    s = train_df[col]

    if str(s.dtype) in ["object", "category", "bool"]:
        categorical.append(col)
    elif col in known_cat_exact:
        categorical.append(col)
    elif s.dropna().isin([0, 1]).all():
        categorical.append(col)
    elif s.dropna().isin([0, 1, 2]).all():
        categorical.append(col)
    else:
        numerical.append(col)

In [ ]:
lgb_train = train_df.copy()
lgb_valid = valid_df.copy()

In [ ]:
for col in numerical:
    med = lgb_train[col].median()
    lgb_train[col] = lgb_train[col].fillna(med)
    lgb_valid[col] = lgb_valid[col].fillna(med)

for col in categorical:
    train_cats = pd.Index(lgb_train[col].astype(str).unique())
    lgb_train[col] = pd.Categorical(lgb_train[col].astype(str), categories=train_cats)
    lgb_valid[col] = pd.Categorical(lgb_valid[col].astype(str), categories=train_cats)

feature_cols = [c for c in train_df.columns if c not in [TARGET] + ID_COLS]

X_train_lgb = lgb_train[feature_cols]
y_train = lgb_train[TARGET].values

X_valid_lgb = lgb_valid[feature_cols]
y_valid = lgb_valid[TARGET].values

print("LightGBM train shape:", X_train_lgb.shape)
print("LightGBM valid shape:", X_valid_lgb.shape)
print("Categorical features:", len(categorical))
print("Numerical features:", len(numerical))


In [ ]:
ID_COLS = [c for c in ["TransactionID"] if c in train_df.columns]
feature_cols = [c for c in train_df.columns if c not in [TARGET] + ID_COLS]

In [ ]:
suspect_keywords = [
    "fraud", "label", "target", "chargeback", "review", "approve", "deny",
    "isfraud", "risk", "score", "cb", "post", "future"
]

suspect_cols = [c for c in df.columns if any(k in c.lower() for k in suspect_keywords)]
suspect_cols

In [ ]:
suspect_keywords = [
    "fraud", "label", "target", "chargeback", "review", "approve", "deny",
    "isfraud", "risk", "score", "cb", "post", "future"
]

suspect_cols = [c for c in df.columns if any(k in c.lower() for k in suspect_keywords)]
suspect_cols

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
import lightgbm as lgb
import numpy as np

# Time-aware CV only on the training data
tscv = TimeSeriesSplit(n_splits=3)

lgb_estimator = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    random_state=SEED,
    n_jobs=-1,
    class_weight="balanced"
)

param_dist = {
    "num_leaves": [31, 63, 127],
    "max_depth": [-1, 6, 10, 14],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "n_estimators": [200, 400, 800],
    "min_child_samples": [20, 50, 100],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_alpha": [0.0, 0.1, 1.0],
    "reg_lambda": [0.0, 0.1, 1.0]
}

lgb_search = RandomizedSearchCV(
    estimator=lgb_estimator,
    param_distributions=param_dist,
    n_iter=20,
    scoring="average_precision",
    cv=tscv,
    verbose=2,
    random_state=SEED,
    n_jobs=-1
)

lgb_search.fit(X_train_lgb, y_train)

print("Best LGBM params:")
print(lgb_search.best_params_)
print("Best CV score (AP):", lgb_search.best_score_)

In [ ]:
best_lgb_params = lgb_search.best_params_

lgb_model = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    random_state=SEED,
    n_jobs=-1,
    class_weight="balanced",
    **best_lgb_params
)

lgb_model.fit(
    X_train_lgb,
    y_train,
    eval_set=[(X_valid_lgb, y_valid)],
    eval_metric="auc",
    callbacks=[
        lgb.early_stopping(100),
        lgb.log_evaluation(100)
    ]
)

valid_pred_lgb = lgb_model.predict_proba(X_valid_lgb)[:, 1]

print("LightGBM ROC-AUC:", roc_auc_score(y_valid, valid_pred_lgb))
print("LightGBM PR-AUC :", average_precision_score(y_valid, valid_pred_lgb))

In [ ]:
# # Train LightGBM
# lgb_model = lgb.LGBMClassifier(
#     objective="binary",
#     boosting_type="gbdt",
#     n_estimators=1200,
#     learning_rate=0.03,
#     num_leaves=64,
#     max_depth=-1,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     reg_alpha=0.1,
#     reg_lambda=1.0,
#     random_state=SEED,
#     n_jobs=-1,
#     class_weight="balanced"
# )

# lgb_model.fit(
#     X_train_lgb,
#     y_train,
#     eval_set=[(X_valid_lgb, y_valid)],
#     eval_metric="auc",
#     categorical_feature=categorical,
#     callbacks=[
#         lgb.early_stopping(100),
#         lgb.log_evaluation(100)
#     ]
# )

# valid_pred_lgb = lgb_model.predict_proba(X_valid_lgb)[:, 1]

# print("LightGBM ROC-AUC:", roc_auc_score(y_valid, valid_pred_lgb))
# print("LightGBM PR-AUC :", average_precision_score(y_valid, valid_pred_lgb))


In [ ]:
cat_cardinality = []
for col in categorical:
    cat_cardinality.append((col, train_df[col].nunique(dropna=False), str(train_df[col].dtype)))

cat_cardinality = sorted(cat_cardinality, key=lambda x: x[1], reverse=True)
cat_cardinality[:50]

In [ ]:
low_num = []
for col in numerical:
    low_num.append((col, train_df[col].nunique(dropna=False), str(train_df[col].dtype)))

low_num = sorted(low_num, key=lambda x: x[1])
low_num[:50]

In [ ]:
small_float_cols = []
for col in train_df.columns:
    if col == "isFraud":
        continue
    s = train_df[col]
    if str(s.dtype) == "float64" and s.nunique(dropna=False) <= 10:
        small_float_cols.append((col, sorted(s.dropna().unique())[:20]))

small_float_cols[:30]

## NN Keras

In [ ]:
# Build NN matrices from the same split
nn_train = train_df.copy()
nn_valid = valid_df.copy()

for col in categorical:
    nn_train[col] = nn_train[col].fillna("missing").astype(str)
    nn_valid[col] = nn_valid[col].fillna("missing").astype(str)

for col in numerical:
    med = nn_train[col].median()
    nn_train[col] = nn_train[col].fillna(med)
    nn_valid[col] = nn_valid[col].fillna(med)

X_train_nn = pd.get_dummies(nn_train[feature_cols], columns=categorical, dummy_na=False)
X_valid_nn = pd.get_dummies(nn_valid[feature_cols], columns=categorical, dummy_na=False)

X_train_nn, X_valid_nn = X_train_nn.align(X_valid_nn, join="left", axis=1, fill_value=0)

scaler = StandardScaler()
existing_num_cols = [c for c in numerical if c in X_train_nn.columns]

X_train_nn[existing_num_cols] = scaler.fit_transform(X_train_nn[existing_num_cols])
X_valid_nn[existing_num_cols] = scaler.transform(X_valid_nn[existing_num_cols])

X_train_nn = X_train_nn.astype("float32")
X_valid_nn = X_valid_nn.astype("float32")

print("NN train shape:", X_train_nn.shape)
print("NN valid shape:", X_valid_nn.shape)


In [ ]:
def binary_focal_loss(alpha=0.25, gamma=2.0):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)

        pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        at = tf.where(tf.equal(y_true, 1), alpha, 1 - alpha)

        loss_val = -at * tf.pow(1.0 - pt, gamma) * tf.math.log(pt)
        return tf.reduce_mean(loss_val)
    return loss


def build_dense_nn(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.30),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.20),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.10),
        layers.Dense(1, activation="sigmoid")
    ])
    return model


In [ ]:
nn_model = build_dense_nn(X_train_nn.shape[1])

nn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=binary_focal_loss(alpha=0.25, gamma=2.0),
    metrics=[
        keras.metrics.AUC(name="auc"),
        keras.metrics.AUC(name="pr_auc", curve="PR")
    ]
)

callbacks = [
    EarlyStopping(monitor="val_auc", patience=5, mode="max", restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_auc", patience=2, factor=0.5, mode="max")
]

history = nn_model.fit(
    X_train_nn,
    y_train,
    validation_data=(X_valid_nn, y_valid),
    epochs=10,
    batch_size=2048,
    callbacks=callbacks,
    verbose=1
)

valid_pred_nn = nn_model.predict(X_valid_nn, batch_size=8192).ravel()

print("NN ROC-AUC:", roc_auc_score(y_valid, valid_pred_nn))
print("NN PR-AUC :", average_precision_score(y_valid, valid_pred_nn))


In [ ]:
alphas = np.arange(0.0, 1.01, 0.05)

alpha_rows = []
for alpha in alphas:
    pred = alpha * valid_pred_lgb + (1 - alpha) * valid_pred_nn
    alpha_rows.append({
        "alpha": alpha,
        "roc_auc": roc_auc_score(y_valid, pred),
        "pr_auc": average_precision_score(y_valid, pred)
    })

alpha_df = pd.DataFrame(alpha_rows).sort_values("pr_auc", ascending=False)
display(alpha_df.head(10))

best_alpha = float(alpha_df.iloc[0]["alpha"])
ensemble_valid_pred = best_alpha * valid_pred_lgb + (1 - best_alpha) * valid_pred_nn

print("Best alpha:", best_alpha)
print("Ensemble ROC-AUC:", roc_auc_score(y_valid, ensemble_valid_pred))
print("Ensemble PR-AUC :", average_precision_score(y_valid, ensemble_valid_pred))


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, balanced_accuracy_score

def evaluate_predictions(y_true, y_prob, threshold=0.5, name="model"):
    y_pred = (y_prob >= threshold).astype(int)

    print(f"\n===== {name} @ threshold={threshold:.2f} =====")
    print("ROC-AUC      :", roc_auc_score(y_true, y_prob))
    print("PR-AUC       :", average_precision_score(y_true, y_prob))
    print("Precision    :", precision_score(y_true, y_pred, zero_division=0))
    print("Recall       :", recall_score(y_true, y_pred, zero_division=0))
    print("F1           :", f1_score(y_true, y_pred, zero_division=0))
    print("Balanced Acc :", balanced_accuracy_score(y_true, y_pred))

evaluate_predictions(y_valid, valid_pred_lgb, name="LightGBM")
evaluate_predictions(y_valid, valid_pred_nn, name="Keras NN")
evaluate_predictions(y_valid, ensemble_valid_pred, name="Ensemble")


In [ ]:
thresholds = np.arange(0.05, 0.96, 0.05)

threshold_rows = []
for t in thresholds:
    y_pred = (ensemble_valid_pred >= t).astype(int)
    threshold_rows.append({
        "threshold": t,
        "precision": precision_score(y_valid, y_pred, zero_division=0),
        "recall": recall_score(y_valid, y_pred, zero_division=0),
        "f1": f1_score(y_valid, y_pred, zero_division=0),
        "balanced_acc": balanced_accuracy_score(y_valid, y_pred)
    })

threshold_df = pd.DataFrame(threshold_rows).sort_values("f1", ascending=False)
display(threshold_df.head(10))

best_threshold = float(threshold_df.iloc[0]["threshold"])
print("Best threshold by F1:", best_threshold)

evaluate_predictions(y_valid, ensemble_valid_pred, threshold=best_threshold, name="Ensemble tuned")


In [ ]:
ARTIFACT_DIR = "./artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

lgb_model.booster_.save_model(os.path.join(ARTIFACT_DIR, "lgbm_model.txt"))
nn_model.save(os.path.join(ARTIFACT_DIR, "keras_nn_model.keras"))


with open(os.path.join(ARTIFACT_DIR, "ensemble_config.json"), "w") as f:
    json.dump(
        {
            "best_alpha": best_alpha,
            "best_threshold": best_threshold,
            "feature_cols": feature_cols,
            "categorical": categorical,
            "numerical": numerical
        },
        f,
        indent=2
    )

print("Saved models and config to:", ARTIFACT_DIR)
